In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, add
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -------------------------------
# 1. Load Pre-trained ResNet Model
# -------------------------------

resnet = ResNet50(weights='imagenet')

# Remove last classification layer
resnet_model = Model(inputs=resnet.input, outputs=resnet.layers[-2].output)

# --------------------------------
# 2. Extract Image Features
# --------------------------------

def extract_features(img_path):

    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    features = resnet_model.predict(img)

    return features


# --------------------------------
# 3. Define Vocabulary (Example)
# --------------------------------

word_to_index = {
    "startseq":1,
    "endseq":2,
    "a":3,
    "man":4,
    "dog":5,
    "running":6,
    "on":7,
    "grass":8
}

index_to_word = {v:k for k,v in word_to_index.items()}

vocab_size = len(word_to_index) + 1
max_length = 10


# --------------------------------
# 4. Build Caption Generator Model
# --------------------------------

# Image feature input
image_input = Input(shape=(2048,))
image_dense = Dense(256, activation='relu')(image_input)

# Text input
text_input = Input(shape=(max_length,))
embedding = Embedding(vocab_size, 256, mask_zero=True)(text_input)
lstm = LSTM(256)(embedding)

# Combine image and text features
decoder = add([image_dense, lstm])
decoder = Dense(256, activation='relu')(decoder)
outputs = Dense(vocab_size, activation='softmax')(decoder)

model = Model(inputs=[image_input, text_input], outputs=outputs)

model.compile(loss='categorical_crossentropy', optimizer='adam')

print(model.summary())


# --------------------------------
# 5. Generate Caption
# --------------------------------

def generate_caption(photo):

    in_text = "startseq"

    for i in range(max_length):

        sequence = [word_to_index[w] for w in in_text.split() if w in word_to_index]
        sequence = pad_sequences([sequence], maxlen=max_length)

        yhat = model.predict([photo, sequence], verbose=0)

        yhat = np.argmax(yhat)

        word = index_to_word.get(yhat)

        if word is None:
            break

        in_text += " " + word

        if word == "endseq":
            break

    return in_text


# --------------------------------
# 6. Test the Model
# --------------------------------

image_path = "test.jpg"

photo_features = extract_features(image_path)

caption = generate_caption(photo_features)

print("Generated Caption:", caption)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)    │ (None, 10)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_4 (InputLayer)    │ (None, 2048)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_1 (Embedding)       │ (None, 10, 256)           │           2,304 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal_1 (NotEqual)        │ (None, 10)                │               0 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 256)               │         524,544 │ input_layer_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_1 (LSTM)                 │ (None, 256)               │         525,312 │ embedding_1[0][0],         │
│                               │                           │                 │ not_equal_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_1 (Add)                   │ (None, 256)               │               0 │ dense_3[0][0],             │
│                               │                           │                 │ lstm_1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_4 (Dense)               │ (None, 256)               │          65,792 │ add_1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_5 (Dense)               │ (None, 9)                 │           2,313 │ dense_4[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,120,265 (4.27 MB)

 Trainable params: 1,120,265 (4.27 MB)

 Non-trainable params: 0 (0.00 B)

None
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Generated Caption: startseq on on on on on on on on on on
